In [1]:
import pandas as pd

pd.set_option('display.max_rows', 128)
pd.set_option('display.max_columns', 128)

In [2]:
def get_hue():
    df = pd.read_parquet("../../data/hue-dataset-hourly.complete.parquet")

    domain_features = {
        'EVs': ['EVs'],
        'measurements': [c for c in df.columns if c.startswith('energy')],
    }

    contextual_features = {
        'geolocation': ['lat', 'lon', 'region', 'country', 'residential_id'],
        'time': [name for name in df.columns if name.startswith('year_') or name.startswith('day_') or name.startswith('week_')] + ['timestamp', 'tz', 'is_dst'],
        'weather_conditions': ['temperature', 'humidity', 'pressure', 'solar_altitude', 'solar_azimuth', 'solar_radiation'],
        'building_properties': ['house_type', 'facing'] + ['RUs', 'SN', 'FAGF', 'HP', 'FPG', 'FPE', 'IFRHG', 'NAC', 'FAC', 'PAC', 'BHE', 'IFRHE', 'WRHIR', 'GEOTH'],
    }

    behavioral_features = {
        'social_activities': ['weekday', 'is_holiday', 'is_weekend'],
        #'cooking': ['kitchen_activity'],
        'work_schedule': [
            'yesterday_ratio', 'yesterday_ratio_ahead', 'last_week_ratio', 'last_week_ratio_ahead', 'yesterday_median_energy', 'house_activity',
            'breakfast_6_9', 'dinner_18_21', 'freetime_17_22', 'lunch_11_15', 'sleep_22_7', 'work_9_17',
        ],
    }

    obtained_features = (
        [x for v in domain_features.values() for x in v] +
        [x for v in contextual_features.values() for x in v] +
        [x for v in behavioral_features.values() for x in v] +
        #["tz", 'region', "country", "timestamp"] + # useless in our case because it is constant
        ['prediction_1h_ahead', 'prediction_3h_ahead']
    )

    # Check for missed features #1, #2, and duplicates
    
    # Sanity check: Check if I missed any feature avaiable in dataset that is not in obtained features
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(obtained_features) - set(df.columns)
    assert not intersection, intersection # Should be empty

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection # Should be empty

    duplicates = set([x for x in obtained_features if obtained_features.count(x) > 1])
    assert not duplicates, duplicates # Should be empty

    #print(set(df.columns) - set(obtained_features), set(obtained_features) - set(df.columns), set([x for x in obtained_features if obtained_features.count(x) > 1]))


    domain_features = [x for v in domain_features.values() for x in v]
    contextual_features = [x for v in contextual_features.values() for x in v]
    behavioral_features = [x for v in behavioral_features.values() for x in v]

    return df, (domain_features, contextual_features, behavioral_features)

get_hue();

In [3]:
def get_refit():
    df = pd.read_parquet("../../data/refit-dataset-hourly.complete.parquet")

    domain_features = {
        'PVs': ['PVs'],
        'measurements': [c for c in df.columns if c.startswith('energy')] + [c for c in df.columns if c.startswith('appliance')],
    }

    contextual_features = {
        'geolocation': ['lat', 'lon', 'country', 'house', 'location'],
        'time': [name for name in df.columns if name.startswith('year_') or name.startswith('day_') or name.startswith('week_')] + ['timestamp', 'tz', 'is_dst'],
        'weather_conditions': ['solar_altitude', 'solar_azimuth', 'solar_radiation'],
        'building_properties': ['house_type', 'house_size', 'occupancy', 'construction_year', 'n_appliances'],
    }

    behavioral_features = {
        'social_activities': ['weekday', 'is_holiday', 'is_weekend', 'bedroom', 'cooking', 'entertainment', 'cleaning', 'personal grooming', 'work at home'],
        '???': ['heating'],
        'work_schedule': [
            'yesterday_ratio', 'yesterday_ratio_ahead', 'yesterday_median_energy', 'last_week_ratio', 'last_week_ratio_ahead',
            'breakfast_6_9', 'dinner_18_21', 'freetime_17_22', 'lunch_11_15', 'sleep_22_7', 'work_9_17',
        ],
    }

    obtained_features = (
        [x for v in domain_features.values() for x in v] +
        [x for v in contextual_features.values() for x in v] +
        [x for v in behavioral_features.values() for x in v] +
        ['prediction_1h_ahead', 'prediction_3h_ahead']
    )

    # Check for missed features #1, #2, and duplicates
    
    # Sanity check: Check if I missed any feature avaiable in dataset that is not in obtained features
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(obtained_features) - set(df.columns)
    assert not intersection, intersection # Should be empty

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection # Should be empty

    duplicates = set([x for x in obtained_features if obtained_features.count(x) > 1])
    assert not duplicates, duplicates # Should be empty

    #print(set(df.columns) - set(obtained_features), set(obtained_features) - set(df.columns), set([x for x in obtained_features if obtained_features.count(x) > 1]))


    domain_features = [x for v in domain_features.values() for x in v]
    contextual_features = [x for v in contextual_features.values() for x in v]
    behavioral_features = [x for v in behavioral_features.values() for x in v]

    return df, (domain_features, contextual_features, behavioral_features)

get_refit();

In [4]:
def get_uciml():
    df = pd.read_parquet("../../data/uciml-dataset-hourly.complete.parquet")

    # Remove energy as it was not used in the original paper (for reproducability)
    #df.drop(columns=[c for c in df.columns if c.startswith("energy")], inplace=True)
        
    domain_features = {
        #'PVs': [],
        #'EVs': [],
        'measurements': sorted([
            name for name in df.columns
            if ('power' in name) or ('meter' in name) or ('volt' in name) or ('intensity' in name) or (name.startswith("energy"))
        ])
    }

    contextual_features = {
        'geolocation': ['lat', 'lon', 'region', 'country'],
        'time': [name for name in df.columns if name.startswith('year_') or name.startswith('day_') or name.startswith('week_')] + ['timestamp', 'tz', 'is_dst'],
        'weather_conditions': [name for name in df.columns if name.startswith('solar_')],
    }

    behavioral_features = {
        'social_activities': ['weekday', 'is_holiday', 'is_weekend'],
        'cooking': ['kitchen_activity'],
        'work_schedule': [
            'yesterday_ratio', 'yesterday_ratio_ahead', 'last_week_ratio', 'last_week_ratio_ahead', 'yesterday_median_energy', 'house_activity',
            'breakfast_6_9', 'dinner_18_21', 'freetime_17_22', 'lunch_11_15', 'sleep_22_7', 'work_9_17',
        ],
    }

    obtained_features = (
        [x for v in domain_features.values() for x in v] +
        [x for v in contextual_features.values() for x in v] +
        [x for v in behavioral_features.values() for x in v] +
        #["tz", 'region', "country", "timestamp"] + # useless in our case because it is constant
        ['prediction_1h_ahead', 'prediction_3h_ahead',]
    )

    # Check for missed features #1, #2, and duplicates
    
    # Sanity check: Check if I missed any feature avaiable in dataset that is not in obtained features
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(obtained_features) - set(df.columns)
    assert not intersection, intersection # Should be empty

    # Sanity check: Check if I missed any feature avaiable in obtained features that is not in dataset
    intersection = set(df.columns) - set(obtained_features)
    assert not intersection, intersection # Should be empty

    duplicates = set([x for x in obtained_features if obtained_features.count(x) > 1])
    assert not duplicates, duplicates # Should be empty

    #print(set(df.columns) - set(obtained_features), set(obtained_features) - set(df.columns), set([x for x in obtained_features if obtained_features.count(x) > 1]))

    domain_features = [x for v in domain_features.values() for x in v]
    contextual_features = [x for v in contextual_features.values() for x in v]
    behavioral_features = [x for v in behavioral_features.values() for x in v]

    return df, (domain_features, contextual_features, behavioral_features)


get_uciml();

In [5]:
df_hue, (hue_domain_features, hue_contextual_features, hue_behavioral_features) = get_hue()
all_hue_features = (hue_domain_features, hue_contextual_features, hue_behavioral_features)
all_hue_features = sum(map(list, all_hue_features), [])


df_refit, (refit_domain_features, refit_contextual_features, refit_behavioral_features) = get_refit()
prev = sum(map(list, (refit_domain_features, refit_contextual_features, refit_behavioral_features)), [])

# Remove data collected from submeters
keywords = [
    "appliance", 'personal grooming', 'cooking', 'cleaning', 'entertainment', 'work at home', 'heating', 'PVs', 'bedroom',
    'breakfast_6_9', 'lunch_11_15', 'dinner_18_21', 'sleep_22_7', 'work_9_17', 'freetime_17_22'
]
#refit_domain_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), refit_domain_features))
#refit_contextual_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), refit_contextual_features))
#refit_behavioral_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), refit_behavioral_features))

refit_all_features = (refit_domain_features, refit_contextual_features, refit_behavioral_features)
refit_all_features = sum(map(list, refit_all_features), [])



df_uciml, (uciml_domain_features, uciml_contextual_features, uciml_behavioral_features) = get_uciml()

#prev = sum(map(list, (uciml_domain_features, uciml_contextual_features, uciml_behavioral_features)), [])

# Remove submeter features
#keywords = ('meter', 'volt', 'kitchen', 'breakfast', 'dinner', 'lunch', 'sleep', 'work', 'freetime')
#uciml_domain_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), uciml_domain_features))
#uciml_contextual_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), uciml_contextual_features))
#uciml_behavioral_features = list(filter(lambda feature: not any([(k in feature) for k in keywords]), uciml_behavioral_features))

uciml_all_features = (uciml_domain_features, uciml_contextual_features, uciml_behavioral_features)
uciml_all_features = sum(map(list, uciml_all_features), [])



print(set(prev) - set(refit_all_features))
print(prev)

set()
['PVs', 'energy', 'energy_mean_3h', 'energy_std_3h', 'energy_yesterday', 'energy_yesterday_ahead', 'energy_last_week', 'energy_last_week_ahead', 'appliance1', 'appliance2', 'appliance3', 'appliance4', 'appliance5', 'appliance6', 'appliance7', 'appliance8', 'appliance9', 'appliance1_mean_3h', 'appliance1_std_3h', 'appliance2_mean_3h', 'appliance2_std_3h', 'appliance3_mean_3h', 'appliance3_std_3h', 'appliance4_mean_3h', 'appliance4_std_3h', 'appliance5_mean_3h', 'appliance5_std_3h', 'appliance6_mean_3h', 'appliance6_std_3h', 'appliance7_mean_3h', 'appliance7_std_3h', 'appliance8_mean_3h', 'appliance8_std_3h', 'appliance9_mean_3h', 'appliance9_std_3h', 'lat', 'lon', 'country', 'house', 'location', 'day_percent', 'week_percent', 'year_percent', 'timestamp', 'tz', 'is_dst', 'solar_altitude', 'solar_azimuth', 'solar_radiation', 'house_type', 'house_size', 'occupancy', 'construction_year', 'n_appliances', 'weekday', 'is_holiday', 'is_weekend', 'bedroom', 'cooking', 'entertainment', 'cle

In [6]:
# Common features found in all datasets
common_domain_features = set.intersection(set(hue_domain_features), set(refit_domain_features), set(uciml_domain_features))
print(f"{common_domain_features=}")

common_contextual_features = set.intersection(set(hue_contextual_features), set(refit_contextual_features), set(uciml_contextual_features))
print(f"{common_contextual_features=}")

common_behavioral_features = set.intersection(set(hue_behavioral_features), set(refit_behavioral_features), set(uciml_behavioral_features))
print(f"{common_behavioral_features=}")

common_all_features = sum(map(list, [common_domain_features, common_contextual_features, common_behavioral_features]), [])


common_domain_features = list(common_domain_features)
common_contextual_features = list(common_contextual_features)
common_behavioral_features = list(common_behavioral_features)

common_domain_features={'energy', 'energy_last_week_ahead', 'energy_std_3h', 'energy_last_week', 'energy_mean_3h', 'energy_yesterday_ahead', 'energy_yesterday'}
common_contextual_features={'tz', 'lat', 'solar_azimuth', 'year_percent', 'day_percent', 'is_dst', 'solar_radiation', 'lon', 'week_percent', 'solar_altitude', 'timestamp', 'country'}
common_behavioral_features={'dinner_18_21', 'yesterday_median_energy', 'work_9_17', 'is_weekend', 'freetime_17_22', 'last_week_ratio', 'lunch_11_15', 'is_holiday', 'weekday', 'last_week_ratio_ahead', 'yesterday_ratio_ahead', 'yesterday_ratio', 'sleep_22_7', 'breakfast_6_9'}


In [7]:
from pprint import pprint

#pprint(uciml_domain_features)


In [8]:
from sklearn import ensemble, tree, model_selection, neural_network, compose, preprocessing, pipeline, metrics
import xgboost as xgb
import shap

transformers = [
    # Drop timestamp features
    ('drop', compose.make_column_selector(dtype_include=['datetime', 'datetimetz'])),
    # One-hot encode categorical or object-type features
    (preprocessing.OneHotEncoder(sparse_output=False), compose.make_column_selector(dtype_include=['category', 'object'])),
]

mapper = compose.make_column_transformer(
    *transformers,
    n_jobs=8,
    remainder='passthrough',
    verbose=True,
    verbose_feature_names_out=False,
)

# Pandas format to keep track of feature names
mapper = mapper.set_output(transform='pandas')

regr = pipeline.make_pipeline(
    mapper,
    xgb.XGBRegressor(n_jobs=-1, random_state=42),
    verbose=True
)

prediction = 'prediction_1h_ahead'

#used_features = list(common_domain_features) + list(common_contextual_features) + list(common_behavioral_features)
df = df_uciml
domain_features = uciml_domain_features
contextual_features = uciml_contextual_features
behavioral_features = uciml_behavioral_features





# EDIT AS NECESSERY !!!
used_features = domain_features + contextual_features + behavioral_features


print(f"{used_features}")
print(f"{len(used_features)=}")

# Remove constant columns
for col in ('region', 'tz', 'country', 'timestamp'):
    if col in used_features:
        used_features.remove(col)

_df = df[[prediction, *used_features]].copy().reset_index(drop=True).dropna()

X, y = _df[used_features], _df[prediction]
print(X.info())


X_train, X_test, y_train, y_test = [], [], [], []

if False:
    for _, group in df.groupby(by='residential_id'):

        _group = df.loc[group.index, [prediction, *used_features]].copy().dropna()

        _X_train, _X_test, _y_train, _y_test = model_selection.train_test_split(_group[used_features], _group[prediction], train_size=0.8, shuffle=False)

        X_train.append(_X_train)
        X_test.append(_X_test)
        y_train.append(_y_train)
        y_test.append(_y_test)

    X_train = pd.concat(X_train)
    X_test = pd.concat(X_test)
    y_train = pd.concat(y_train)
    y_test = pd.concat(y_test)
else:
    X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, train_size=0.8, shuffle=False)

print(len(X_train), len(X_test))

y_pred = regr.fit(X_train, y_train).predict(X_test)

mse = metrics.mean_squared_error(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)


print(f'& {mse:,.3f} & {mape:,.3f}  % 2023-04-19 -- XGB')

/home/gcerar/miniconda3/envs/feature-store/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['energy', 'energy_last_week', 'energy_last_week_ahead', 'energy_mean_3h', 'energy_std_3h', 'energy_yesterday', 'energy_yesterday_ahead', 'global_active_power', 'global_active_power_last_week', 'global_active_power_last_week_ahead', 'global_active_power_mean_3h', 'global_active_power_std_3h', 'global_active_power_yesterday', 'global_active_power_yesterday_ahead', 'global_intensity', 'global_intensity_last_week', 'global_intensity_last_week_ahead', 'global_intensity_mean_3h', 'global_intensity_std_3h', 'global_intensity_yesterday', 'global_intensity_yesterday_ahead', 'global_reactive_power', 'global_reactive_power_last_week', 'global_reactive_power_last_week_ahead', 'global_reactive_power_mean_3h', 'global_reactive_power_std_3h', 'global_reactive_power_yesterday', 'global_reactive_power_yesterday_ahead', 'sub_metering_1', 'sub_metering_1_last_week', 'sub_metering_1_last_week_ahead', 'sub_metering_1_mean_3h', 'sub_metering_1_std_3h', 'sub_metering_1_yesterday', 'sub_metering_1_yesterday_

In [9]:
import shap

_X = mapper.fit_transform(X)

# Rename columns to to show what group in belongs
rename = {}
for col in _X.columns:
    for sample in domain_features:
        if col.startswith(sample):
            rename[col] = f'{col} (D)'

    for sample in contextual_features:
        if col.startswith(sample):
            rename[col] = f'{col} (C)'

    for sample in behavioral_features:
        if col.startswith(sample):
            rename[col] = f'{col} (B)'

_X.rename(columns=rename, inplace=True)
model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
model.fit(_X, y)

[ColumnTransformer] ..... (1 of 1) Processing remainder, total=   0.0s


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [10]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(_X)

In [11]:
import numpy as np

def shap_feature_ranking(X, shap_values):
    feature_names = list(X.columns)

    vals = np.abs(shap_values.values).mean(0)

    feature_importance = pd.DataFrame(
        data=list(zip(feature_names, vals)),
        columns=['col_name', 'feature_importance_vals'],
    )
    feature_importance.sort_values(
        by=['feature_importance_vals'],
        ascending=False,
        inplace=True,
    )
    
    return feature_importance

shap_feature_ranking(_X, shap_values)

,col_name,feature_importance_vals
14,global_intensity (D),0.326349
49,unmetered (D),0.177883
42,sub_metering_3 (D),0.069541
65,day_percent (C),0.061128
45,sub_metering_3_mean_3h (D),0.049124
28,sub_metering_1 (D),0.048729
7,global_active_power (D),0.047745
55,unmetered_yesterday_ahead (D),0.042864
51,unmetered_last_week_ahead (D),0.040236
39,sub_metering_2_std_3h (D),0.034304


In [12]:
total_sum = {'D': 0, 'C': 0, 'B': 0}
count = {'D': 0, 'C': 0, 'B': 0}

n_features = len(_X.columns)

for feature, value in zip(_X.columns, np.mean(np.abs(shap_values.values), axis=0)):
    if feature.endswith('(D)'):
        total_sum['D'] += value
        count['D'] += 1

    elif feature.endswith('(C)'):
        total_sum['C'] += value
        count['C'] += 1

    elif feature.endswith('(B)'):
        total_sum['B'] += value
        count['B'] += 1

assert n_features == sum(count.values())

# Normalize
norm_cum_shap_contrib = { key: 100 * (total_sum[key] / sum(total_sum.values())) for key in ['D', 'C', 'B'] }

print(f"{count=} {sum(count.values())=}")
print(f"{total_sum=}")
print(f"{norm_cum_shap_contrib=}")


print(sum(total_sum.values()))


count={'D': 63, 'C': 9, 'B': 16} sum(count.values())=88
total_sum={'D': 1.3472882900387049, 'C': 0.15914731845259666, 'B': 0.10076349272276275}
norm_cum_shap_contrib={'D': 83.82833769761163, 'C': 9.902153275992886, 'B': 6.269509026395477}
1.6071991012140643


In [13]:
import math

# Account only top 10%
results = shap_feature_ranking(_X, shap_values)
results = results.sort_values("feature_importance_vals", ascending=False)

results["category"] = results["col_name"].apply(lambda x: x[-2:-1])
results["col_name"] = results["col_name"].apply(lambda x: x[:-4])

subset = []

for idx, group in results.groupby("category"):
    n_items = round(len(group) * 0.10)
    print(n_items)
    if n_items == 0: n_items = 1
    sgroup = group.head(n_items)
    subset.append(sgroup)

subset = pd.concat(subset)
subset

top_features = list(subset["col_name"])


__X = mapper.fit_transform(X[top_features])

# Rename columns to to show what group in belongs
rename = {}
for col in __X.columns:
    for sample in domain_features:
        if col.startswith(sample):
            rename[col] = f'{col} (D)'

    for sample in contextual_features:
        if col.startswith(sample):
            rename[col] = f'{col} (C)'

    for sample in behavioral_features:
        if col.startswith(sample):
            rename[col] = f'{col} (B)'

__X.rename(columns=rename, inplace=True)
model = xgb.XGBRegressor(random_state=42, n_jobs=-1, )
model.fit(__X, y)


shap_feature_ranking(__X, shap_values)


total_sum = {'D': 0, 'C': 0, 'B': 0}
count = {'D': 0, 'C': 0, 'B': 0}

n_features = len(__X.columns)

for feature, value in zip(__X.columns, np.mean(np.abs(shap_values.values), axis=0)):
    if feature.endswith('(D)'):
        total_sum['D'] += value
        count['D'] += 1

    elif feature.endswith('(C)'):
        total_sum['C'] += value
        count['C'] += 1

    elif feature.endswith('(B)'):
        total_sum['B'] += value
        count['B'] += 1

assert n_features == sum(count.values())

# Normalize
norm_cum_shap_contrib = { key: 100 * (total_sum[key] / sum(total_sum.values())) for key in ['D', 'C', 'B'] }

print(f"{count=} {sum(count.values())=}")
print(f"{total_sum=}")
print(f"{norm_cum_shap_contrib=}")

print(list(__X.columns))
print(sum(total_sum.values()))

2
1
6
[ColumnTransformer] ..... (1 of 1) Processing remainder, total=   0.0s
count={'D': 6, 'C': 1, 'B': 2} sum(count.values())=9
total_sum={'D': 0.08119617449119687, 'C': 0.007849463261663914, 'B': 0.026949932798743248}
norm_cum_shap_contrib={'D': 69.99937506671806, 'C': 6.767037072481876, 'B': 23.233587860800064}
['breakfast_6_9 (B)', 'dinner_18_21 (B)', 'day_percent (C)', 'global_intensity (D)', 'unmetered (D)', 'sub_metering_3 (D)', 'sub_metering_3_mean_3h (D)', 'sub_metering_1 (D)', 'global_active_power (D)']
0.11599557055160403
